# utility Functions

In [10]:
import re
from youtube_transcript_api import YouTubeTranscriptApi

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langsmith import traceable
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [11]:

def load_transcript(url: str) -> str | None:

    pattern = r'(?:v=|\/)([0-9A-Za-z_-]{11})'
    match = re.search(pattern, url)
    if match:
        video_id = match.group(1)
        try:
            captions = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'hi']).snippets
            data = [f"{item.text} ({item.start})" for item in captions]
            return " ".join(data)
        except Exception as e:
            print(f" Error fetching transcript: {e}")
            return None

### text splitter & retriever

In [ ]:
def text_splitter(transcript: str):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    return splitter.create_documents([transcript])
def build_hybrid_retriever(chunks, *, faiss_k: int = 4, bm25_k: int = 4, top_n: int = 5):
    """
    Builds and returns hybrid_retrieve(query) callable.

    Parameters
    ----------
    chunks  : list[Document]  from text_splitter()
    faiss_k : top-k docs fetched from FAISS
    bm25_k  : top-k docs fetched from BM25
    top_n   : final docs returned after merge
    """

    # ── Dense retriever (FAISS) ──────────────────────────────
    embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
    vector_store = FAISS.from_documents(chunks, embeddings)
    
    faiss_retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": faiss_k},
    )

    # ── Sparse retriever (BM25) ──────────────────────────────
    bm25_retriever   = BM25Retriever.from_documents(chunks)
    bm25_retriever.k = bm25_k

    # ── Hybrid callable ──────────────────────────────────────
    
    def hybrid_retrieve(query: str):

        faiss_docs = faiss_retriever.invoke(query)
        bm25_docs  = bm25_retriever.invoke(query)

        print(f"\n{'='*60}")
        print(f" Query: {query}")
        print(f"{'='*60}")

        print(f"\n FAISS docs ({len(faiss_docs)}):")
        for i, doc in enumerate(faiss_docs, 1):
            print(f"  [{i}] {doc.page_content[:120]}...")

        print(f"\n BM25 docs ({len(bm25_docs)}):")
        for i, doc in enumerate(bm25_docs, 1):
            print(f"  [{i}] {doc.page_content[:120]}...")

        # Merge — deduplicate by content, FAISS first then BM25
        seen   = set()
        merged = []
        for doc in faiss_docs + bm25_docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                merged.append(doc)

        final = merged[:top_n]

        print(f"\n Final merged docs ({len(final)}):")
        for i, doc in enumerate(final, 1):
            print(f"  [{i}] {doc.page_content[:120]}...")
        print(f"{'='*60}\n")

        return final

    return hybrid_retrieve

In [13]:
transcripts = load_transcript("https://youtu.be/pJdMxwXBsk0?list=PLKnIA16_RmvaTbihpo4MtzVm4XOQa0ER0")
transcripts

'हाय गाइस, माय नेम इज नितेश एंड यू वेलकम (0.32) टू माय YouTube चैनल। इस वीडियो में भी हम (2.639) लोग अपना लैंग चेन प्लेलिस्ट कंटिन्यू (4.88) करेंगे। और आज का वीडियो का टॉपिक है (6.64) रिट्रीवर्स जो कि एक बहुत इंपॉर्टेंट (9.4) टॉपिक है। अगर आप रैग की बात करो। अगर आप (11.679) एक रैग बेस्ड एप्लीकेशन बनाना चाहते हो तो (14.0) वहां पे रिट्रीवर एक बहुत इंपॉर्टेंट (16.48) कंपोनेंट है। इनफैक्ट अ फ्यूचर में जब आप (18.56) थोड़े एडवांस्ड रैग सिस्टम्स बनाओगे तो (21.359) वहां पे अलग-अलग टाइप के रिट्रीवर्स के (24.08) साथ आप काम करोगे तो उस सेंस में ये जो (26.16) पर्टिकुलर वीडियो है बहुत इंपॉर्टेंट है। (28.64) एंड आई वुड लाइक कि आप इस वीडियो को एंड (30.96) टू एंड देखो। सो आज की वीडियो में मैं (33.28) आपको नॉट ओनली समझाऊंगा कि रिट्रीवर्स (35.92) क्या होते हैं? उनकी जरूरत क्या है? बट एट (38.16) द सेम टाइम मैं आपको अलग-अलग टाइप के (40.8) रिट्रीवर्स के बारे में भी बताऊंगा और जो (42.96) कोड है वो लिख के दिखाऊंगा। ठीक है? सो या (45.6) लेट्स स्टार्ट द वीडियो। सो गाइस वीडियो (49.12) को शुरू करने के पहले मैं आ